# Lab 2 — Ridge, Lasso and Cross-Validation
**Machine Learning I · PEU-CD 2026 · ENEI**

Companion to `tutorial.pdf`. Conventions: Lecture 3's ridge is `Ridge(alpha=lam)`; Lecture 3's lasso
$\tfrac12\|y-X\beta\|^2+\lambda\|\beta\|_1$ is `Lasso(alpha=lam/N)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.linear_model import Ridge, Lasso, LogisticRegression, lasso_path
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_squared_error, accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, roc_auc_score)
np.set_printoptions(precision=4, suppress=True)

X_raw, y_raw = load_diabetes(return_X_y=True)
N, p = X_raw.shape
X = (X_raw - X_raw.mean(0)) / X_raw.std(0)      # standardized columns
y = y_raw - y_raw.mean()                         # centred response -> no intercept needed
print(X.shape, "column means ~0:", np.allclose(X.mean(0), 0), " column sds = 1:", np.allclose(X.std(0), 1))

## 1. Ridge from the formula
### Task 1 — closed form

In [ ]:
def ridge_closed_form(X, y, lam):
    p = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)

for lam in [0.1, 1, 10, 100]:
    mine = ridge_closed_form(X, y, lam)
    sk = Ridge(alpha=lam, fit_intercept=False).fit(X, y).coef_
    print(f"lam={lam:6}: max |diff| = {np.abs(mine - sk).max():.2e}")
    assert np.allclose(mine, sk, atol=1e-8)

### Task 2 — the SVD picture

In [ ]:
U, d, Vt = np.linalg.svd(X, full_matrices=False)
lam = 10
shrink = d**2 / (d**2 + lam)
yhat_svd = U @ (shrink * (U.T @ y))
yhat_ridge = X @ ridge_closed_form(X, y, lam)
print("SVD formula matches closed form:", np.allclose(yhat_svd, yhat_ridge))
print("singular values d_j :", d)
print("shrinkage factors   :", shrink)
j = np.argmin(shrink); print(f"most shrunk direction: j={j}, d_j={d[j]:.3f}, factor={shrink[j]:.3f}")

lams = np.logspace(-2, 4, 200)
df = np.array([(d**2 / (d**2 + l)).sum() for l in lams])
plt.semilogx(lams, df); plt.xlabel("lambda"); plt.ylabel("df(lambda)"); plt.show()
print("df(0) =", (d**2 / (d**2 + 0)).sum())

### Task 3 — the ridge path

In [ ]:
coefs = np.array([ridge_closed_form(X, y, l) for l in lams])
plt.semilogx(lams, coefs); plt.xlabel("lambda"); plt.ylabel("coefficients"); plt.title("ridge path"); plt.show()
# In the SVD basis beta_lam = V diag(d_j/(d_j^2+lam)) U^T y: each coordinate of V^T beta is the OLS value
# times a strictly positive factor, so V^T beta never crosses zero; beta itself can only be zero where
# a linear combination of shrunk OLS coordinates happens to cancel, which is a measure-zero event in lam.
print("min |beta_j| over the path (never exactly 0):", np.abs(coefs).min())

## 2. Lasso from the formula
### Task 4 — soft thresholding

In [ ]:
def soft(b, lam):
    b = np.asarray(b, dtype=float)
    return np.sign(b) * np.maximum(np.abs(b) - lam, 0.0)

print("lasso :", soft([3, 0.5, -2], 1))
print("ridge :", np.array([3, 0.5, -2]) / (1 + 1))
assert np.allclose(soft([3, 0.5, -2], 1), [2, 0, -1])
assert np.allclose(np.array([3, 0.5, -2]) / 2, [1.5, 0.25, -1])

### Task 5 — coordinate descent

In [ ]:
def lasso_cd(X, y, lam, iters=20000, tol=1e-10):
    n, p = X.shape
    beta = np.zeros(p)
    r = y.copy()
    for it in range(iters):
        max_change = 0.0
        for j in range(p):
            r_j = r + X[:, j] * beta[j]
            b_new = soft(X[:, j] @ r_j, lam) / (X[:, j] @ X[:, j])
            r = r_j - X[:, j] * b_new
            max_change = max(max_change, abs(b_new - beta[j])); beta[j] = b_new
        if max_change < tol:          # correlated columns make plain CD slow; sweep until nothing moves
            break
    return beta

for lam in [50, 200, 800]:
    mine = lasso_cd(X, y, lam)
    sk = Lasso(alpha=lam / N, fit_intercept=False, max_iter=100000, tol=1e-10).fit(X, y).coef_
    r = y - X @ mine
    corr = X.T @ r
    active = np.abs(mine) > 1e-8
    print(f"lam={lam:4}: max|diff vs sklearn|={np.abs(mine - sk).max():.1e}  zeros={int((~active).sum())}/{p}"
          f"  KKT active max|x'r|-lam={np.abs(np.abs(corr[active]) - lam).max():.1e}"
          f"  inactive max|x'r|={np.abs(corr[~active]).max() if (~active).any() else 0:.1f} <= {lam}")
    assert np.allclose(mine, sk, atol=1e-4)

### Task 6 — the lasso path and $\lambda_{\max}$

In [ ]:
lam_max = np.abs(X.T @ y).max()
print(f"lam_max = {lam_max:.2f}")
print("all zero at 1.01*lam_max:", np.allclose(lasso_cd(X, y, 1.01 * lam_max), 0))
print("not all zero at 0.99*lam_max:", not np.allclose(lasso_cd(X, y, 0.99 * lam_max), 0))

alphas, coefs_l, _ = lasso_path(X, y, n_alphas=100)          # alphas are lam/N
names = load_diabetes().feature_names
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for j in range(p):
    ax[0].plot(alphas * N, coefs_l[j], label=names[j])
ax[0].set_xscale("log"); ax[0].invert_xaxis(); ax[0].set_title("lasso path"); ax[0].legend(fontsize=7, ncol=2)
ax[1].semilogx(lams, coefs); ax[1].invert_xaxis(); ax[1].set_title("ridge path")
plt.show()
order = [names[j] for j in np.argsort([np.argmax(np.abs(coefs_l[j]) > 1e-8) for j in range(p)])]
print("order of entry (decreasing lambda):", order)

## 3. Choosing lambda
### Task 7 — cross-validation with error bars

In [ ]:
grid = np.logspace(-1, 3.2, 40)                     # lam values (Lecture 3 scale)
kf = KFold(10, shuffle=True, random_state=155)
cv_mean, cv_se, nnz = [], [], []
for lam in grid:
    pipe = make_pipeline(StandardScaler(), Lasso(alpha=lam / (0.9 * N), max_iter=50000))   # 9/10 of N rows per fold
    scores = -cross_val_score(pipe, X_raw, y_raw, cv=kf, scoring="neg_mean_squared_error")
    cv_mean.append(scores.mean()); cv_se.append(scores.std(ddof=1) / np.sqrt(10))
    nnz.append(int((np.abs(pipe.fit(X_raw, y_raw)[-1].coef_) > 1e-8).sum()))
cv_mean, cv_se = np.array(cv_mean), np.array(cv_se)
i_min = int(np.argmin(cv_mean))
i_1se = int(np.max(np.where(cv_mean <= cv_mean[i_min] + cv_se[i_min])[0]))   # largest lam within one SE
plt.errorbar(grid, cv_mean, yerr=cv_se, fmt="o-", ms=3); plt.xscale("log")
plt.axvline(grid[i_min], ls="--", label=f"lam_min={grid[i_min]:.1f}, nnz={nnz[i_min]}")
plt.axvline(grid[i_1se], ls=":", label=f"lam_1se={grid[i_1se]:.1f}, nnz={nnz[i_1se]}")
plt.xlabel("lambda"); plt.ylabel("10-fold CV MSE"); plt.legend(); plt.show()

### Task 8 — optimism, measured

In [ ]:
beta_ols = np.linalg.solve(X.T @ X, X.T @ y)
sigma2 = ((y - X @ beta_ols) ** 2).sum() / (N - p - 1)
print(f"sigma^2 (OLS) = {sigma2:.1f}")
print(f"{'lam':>6} {'train':>9} {'10-CV':>9} {'CV-train':>9} {'2s2df/N':>9}")
for lam in [0.1, 1, 10, 100, 1000]:
    tr = mean_squared_error(y, X @ ridge_closed_form(X, y, lam))
    cv = -cross_val_score(Ridge(alpha=lam, fit_intercept=False), X, y, cv=kf, scoring="neg_mean_squared_error").mean()
    dfl = (d**2 / (d**2 + lam)).sum()
    print(f"{lam:6} {tr:9.1f} {cv:9.1f} {cv - tr:9.1f} {2 * sigma2 * dfl / N:9.1f}")

## 4. Regularized logistic regression
### Task 9 — separable data has no MLE

In [ ]:
rng = np.random.default_rng(155)
A = rng.normal([-2, 0], 1, (30, 2)); B = rng.normal([2, 0], 1, (30, 2))
Xs = np.vstack([A, B]); ys = np.r_[np.zeros(30), np.ones(30)]
import warnings; warnings.filterwarnings("ignore")
unreg = LogisticRegression(penalty=None, max_iter=10000).fit(Xs, ys)
print("unpenalized: ||w|| =", np.linalg.norm(unreg.coef_).round(2), " min/max fitted prob:",
      unreg.predict_proba(Xs)[:, 1].min().round(4), unreg.predict_proba(Xs)[:, 1].max().round(4))
Cs = [1e3, 1e2, 10, 1, 0.1]
norms = [np.linalg.norm(LogisticRegression(C=C, max_iter=10000).fit(Xs, ys).coef_) for C in Cs]
plt.semilogx(Cs, norms, "o-"); plt.xlabel("C = 1/lambda"); plt.ylabel("||w||"); plt.show()
# As C grows the penalty vanishes and ||w|| grows without bound: the likelihood keeps improving along
# c * w_0 for any separating w_0 (Lecture 2). Any finite lambda pins it down.

### Task 10 — a real classifier, evaluated properly

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True)          # y=1 is benign in sklearn; make malignant the positive class
yc = 1 - yc
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=155, stratify=yc)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
gs = GridSearchCV(pipe, {"logisticregression__C": np.logspace(-3, 2, 11)}, cv=5, scoring="neg_log_loss").fit(Xtr, ytr)
print("chosen C:", gs.best_params_["logisticregression__C"])
prob = gs.predict_proba(Xte)[:, 1]; pred = (prob >= 0.5).astype(int)
print(f"acc={accuracy_score(yte, pred):.3f} prec={precision_score(yte, pred):.3f} rec={recall_score(yte, pred):.3f} F1={f1_score(yte, pred):.3f}")
print("confusion (rows=true 0/1, cols=pred 0/1):\n", confusion_matrix(yte, pred))

fpr, tpr, _ = roc_curve(yte, prob)
auc_sk = roc_auc_score(yte, prob)
auc_pairs = np.mean(prob[yte == 1][:, None] > prob[yte == 0][None, :])
plt.plot(fpr, tpr, label=f"AUC={auc_sk:.4f}"); plt.plot([0, 1], [0, 1], "k--"); plt.legend(); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.show()
print(f"AUC sklearn={auc_sk:.6f}  AUC pairwise={auc_pairs:.6f}")
assert np.isclose(auc_sk, auc_pairs)

c_fp, c_fn = 1, 10
tau = c_fp / (c_fp + c_fn)
for t in [0.5, tau]:
    pr = (prob >= t).astype(int); cm = confusion_matrix(yte, pr)
    cost = c_fp * cm[0, 1] + c_fn * cm[1, 0]
    print(f"threshold={t:.3f}: FP={cm[0,1]} FN={cm[1,0]} expected cost={cost}")

## 5. Exercises — see `tutorial.pdf`, Section 6